# 00. GK-2A v2 데이터 점검 (시간 집계 + zenith)

**목적**: `data/gk2a_v2/` 시간 집계 데이터의 구조·커버리지·품질을 점검하고, 후속 EDA·모델 학습에서 사용할 **mask 정책**을 확정.

**v2 스키마** (`src/preprocess/aggregate_gk2a_hourly.py` 산출):

| 컬럼 | 의미 |
|---|---|
| `datetime_kst` | hour-ending 라벨 (라벨 N = `[N-1:00, N:00)` 평균) |
| `site` | 사이트명 (좌표 중복 collapse 후 11개) |
| `dsr_mean` | 6 슬롯 중 valid한 DSR의 평균 (NaN if all 6 invalid) |
| `dsr_n_valid` | 0~6, 그 시간 valid 슬롯 수 |
| `n_slots` | 보통 6, boundary에서 1~5 |
| `zenith_center` | 시간 중심(라벨 - 30분) 기준 태양 천정각 (pvlib) |
| `lat`, `lon` | 사이트 좌표 |

**제거된 컬럼** (v1 검증 결과 redundant):
- `dsr_dqf`: `dsr.notna()` 와 100% 동치 → 정보 0
- `sw_dqf`: zenith로 99.98% 재현 가능 (SZA>70° 단일 신호로 환원)
- `asr`, `rsr`: PV 예측에 직접 기여 0, 선택적 보존 옵션 폐기

In [1]:
import sys
from pathlib import Path
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DATA_DIR = Path('../../data/gk2a_v2')
files = sorted(DATA_DIR.glob('*.csv'))
print(f'CSV 파일 수: {len(files)}')
print(f'첫: {files[0].name}, 끝: {files[-1].name}')

CSV 파일 수: 49
첫: 202201.csv, 끝: 202601.csv


## 1. 전체 로드 + 컬럼 검증

In [2]:
df = pd.concat(
    [pd.read_csv(f, parse_dates=['datetime_kst']) for f in files],
    ignore_index=True,
)
print(f'총 행: {len(df):,}')
print(f'기간: {df.datetime_kst.min()} ~ {df.datetime_kst.max()}')
print(f'사이트 수: {df.site.nunique()}')
print()
print('컬럼·dtype:')
print(df.dtypes)

총 행: 385,495
기간: 2022-01-01 10:00:00 ~ 2026-01-01 09:00:00
사이트 수: 11

컬럼·dtype:
datetime_kst     datetime64[us]
site                        str
dsr_mean                float64
dsr_n_valid               int64
n_slots                   int64
zenith_center           float64
lat                     float64
lon                     float64
dtype: object


## 2. 사이트별 행 수 + 좌표

In [3]:
site_summary = df.groupby('site').agg(
    rows=('datetime_kst', 'size'),
    lat=('lat', 'first'),
    lon=('lon', 'first'),
    kst_start=('datetime_kst', 'min'),
    kst_end=('datetime_kst', 'max'),
    dsr_notna=('dsr_mean', lambda s: s.notna().sum()),
    dsr_isna=('dsr_mean', lambda s: s.isna().sum()),
)
site_summary['dsr_notna_pct'] = (site_summary['dsr_notna'] / site_summary['rows'] * 100).round(2)
site_summary

,rows,lat,lon,kst_start,kst_end,dsr_notna,dsr_isna,dsr_notna_pct
site,,,,,,,,
경상대,35045,35.18,128.10,2022-01-01 10:00:00,2026-01-01 09:00:00,16262,18783,46.40
고흥만수상,35045,34.57,127.30,2022-01-01 10:00:00,2026-01-01 09:00:00,16273,18772,46.43
광양항세방,35045,34.93,127.71,2022-01-01 10:00:00,2026-01-01 09:00:00,16267,18778,46.42
구미,35045,36.13,128.34,2022-01-01 10:00:00,2026-01-01 09:00:00,16242,18803,46.35
삼천포,35045,34.95,128.07,2022-01-01 10:00:00,2026-01-01 09:00:00,16266,18779,46.41
여수,35045,34.74,127.74,2022-01-01 10:00:00,2026-01-01 09:00:00,16270,18775,46.43
영동,35045,37.18,128.46,2022-01-01 10:00:00,2026-01-01 09:00:00,16222,18823,46.29
영흥,35045,37.26,126.46,2022-01-01 10:00:00,2026-01-01 09:00:00,16105,18940,45.96
예천,35045,36.65,128.46,2022-01-01 10:00:00,2026-01-01 09:00:00,16233,18812,46.32


→ **모델 함의**: 사이트별 행 수 일관성 = 시간 격자 통일. PV 학습 시 사이트별 데이터 균형 자동 확보. dsr_notna_pct (~42%)는 한국 위도 기준 일조 가능 시간 비율과 일치 (24h 중 ~10h 한낮 + dawn/dusk fade).

## 3. zenith × dsr_mean 분포 — 핵심 진단

**판독 기준**: zenith가 작을수록(태양 높을수록) DSR 커야 함 (clearsky 한계 ~1000 W/m²). zenith>90°는 야간 → DSR=NaN.

In [4]:
# zenith 버킷 (10° 단위)
df['zenith_bucket'] = pd.cut(
    df.zenith_center,
    bins=[0, 30, 60, 70, 80, 86, 90, 96, 180],
    labels=['0-30°', '30-60°', '60-70°', '70-80°', '80-86°', '86-90°', '90-96°', '>96°'],
)
g = df.groupby('zenith_bucket', observed=False).agg(
    n=('dsr_mean', 'size'),
    dsr_notna=('dsr_mean', lambda s: s.notna().sum()),
    dsr_min=('dsr_mean', 'min'),
    dsr_mean_avg=('dsr_mean', 'mean'),
    dsr_max=('dsr_mean', 'max'),
)
g['notna_pct'] = (g['dsr_notna'] / g['n'] * 100).round(1)
g.round(2)

,n,dsr_notna,dsr_min,dsr_mean_avg,dsr_max,notna_pct
zenith_bucket,,,,,,
0-30°,23682,23682,12.38,666.67,1020.57,100.0
30-60°,78258,78258,0.00,502.06,922.62,100.0
60-70°,33870,33870,0.00,334.31,580.73,100.0
70-80°,29442,29442,0.00,218.14,384.20,100.0
80-86°,15488,13366,0.00,170.49,295.20,86.3
86-90°,12570,41,0.00,142.07,195.20,0.3
90-96°,16463,0,NaN,NaN,NaN,0.0
>96°,175722,0,NaN,NaN,NaN,0.0


→ **모델 함의**: zenith 단조 관계 확인. zenith<86° 는 거의 100% DSR notna. 86~90° 는 transition (일부 측정 살아있음). 90° 이상은 야간 (DSR NaN). 이 패턴이 깨지면 위성 또는 zenith 계산 버그 의심.

## 4. dsr_n_valid 분포 — 시간 집계 완전성

In [5]:
print('--- dsr_n_valid 전체 분포 ---')
print(df.dsr_n_valid.value_counts().sort_index())
print()
print('--- zenith bucket × dsr_n_valid ---')
print(df.groupby('zenith_bucket', observed=False)['dsr_n_valid'].value_counts().unstack(fill_value=0))

--- dsr_n_valid 전체 분포 ---
dsr_n_valid
0     206836
1       4286
2       6588
3       5969
4       7067
5      18212
6     136449
10         2
12        86
Name: count, dtype: int64

--- zenith bucket × dsr_n_valid ---
dsr_n_valid        0     1     2     3     4     5      6   10  12
zenith_bucket                                                     
0-30°               0    11    11    22   179   309  23150   0   0
30-60°              0    33    55    55   433  4750  72888   0  44
60-70°              0     0    11    22   123  3709  30005   0   0
70-80°              0     2   113  3118  6315  9444  10406   2  42
80-86°           2122  4199  6398  2752    17     0      0   0   0
86-90°          12529    41     0     0     0     0      0   0   0
90-96°          16463     0     0     0     0     0      0   0   0
>96°           175722     0     0     0     0     0      0   0   0


→ **모델 함의**: zenith<86° 한낮은 거의 항상 n_valid=6 (정상 측정). 86~90° transition에서 n_valid 1~5 mix (위성이 늦게 끊거나 일부 슬롯만). 90° 이상은 n_valid=0 일관. n_valid<6인 한낮 행은 위성 부분 결손 → 통계적으로 적은 슬롯 평균이라 분산 ↑.

## 5. 한낮 sat_outage 검증 (희귀 케이스)

**판독 기준**: zenith<86°인데 dsr_mean=NaN인 행 = 위성이 한낮 측정 못 한 케이스. 1% 미만이어야 정상.

In [6]:
outage = df[(df.zenith_center < 86) & df.dsr_mean.isna()]
day_total = (df.zenith_center < 86).sum()
print(f'한낮(zenith<86°) 전체: {day_total:,}')
print(f'  그중 sat_outage (dsr=NaN): {len(outage):,} ({len(outage)/day_total*100:.3f}%)')
print()
if len(outage) > 0:
    print('--- sat_outage 사이트·연도별 ---')
    out_sum = outage.groupby([outage.datetime_kst.dt.year, 'site']).size().unstack(fill_value=0)
    print(out_sum)

한낮(zenith<86°) 전체: 180,740
  그중 sat_outage (dsr=NaN): 2,122 (1.174%)

--- sat_outage 사이트·연도별 ---
site          경상대  고흥만수상  광양항세방  구미  삼천포  여수  영동  영흥  예천  창원  탑선
datetime_kst                                                    
2022           42     44     44  43   44  45  43  87  44  43  46
2023           44     43     43  43   44  44  43  89  43  43  46
2024           44     42     43  44   45  44  44  91  44  42  44
2025           46     46     44  47   45  45  47  89  46  45  44
2026            0      0      0   0    0   0   0   1   0   0   0


→ **모델 함의**: sat_outage 비율이 1% 미만이면 무시 가능 (drop 또는 mask). 특정 사이트·연도에 집중되면 위성 알고리즘 또는 픽셀 좌표 이슈 의심 — 별도 진단 필요.

## 6. 같은 timestamp 사이트별 zenith 차이 검증

**판독 기준**: 같은 KST 시각에 11 사이트의 zenith가 lat/lon 차이로 분리돼야 함. 일출/일몰 경계에서 사이트별 day/twilight 갈림 확인.

In [7]:
# 겨울 일몰 부근 (1월 18시) — 사이트별 zenith 비교
ts_check = pd.Timestamp('2024-01-15 18:00:00')
snap = df[df.datetime_kst == ts_check][['site', 'lat', 'lon', 'zenith_center', 'dsr_mean', 'dsr_n_valid']].sort_values('zenith_center')
print(f'=== {ts_check} 시점 사이트별 ===')
print(snap.to_string(index=False))
print()
# 일몰(zenith>90°) 진입한 사이트와 아직 dusk 사이트 분리
set_count = (snap.zenith_center >= 90).sum()
dusk_count = ((snap.zenith_center >= 86) & (snap.zenith_center < 90)).sum()
day_count = (snap.zenith_center < 86).sum()
print(f'분류: daylight {day_count} / dusk {dusk_count} / 일몰 후 {set_count}')

=== 2024-01-15 18:00:00 시점 사이트별 ===
 site   lat    lon  zenith_center  dsr_mean  dsr_n_valid
   탑선 35.24 126.81      88.480693       NaN            0
고흥만수상 34.57 127.30      88.530047       NaN            0
   여수 34.74 127.74      88.929703       NaN            0
광양항세방 34.93 127.71      88.993839       NaN            0
   영흥 37.26 126.46      89.166678       NaN            0
  삼천포 34.95 128.07      89.266171       NaN            0
  경상대 35.18 128.10      89.391577       NaN            0
   창원 35.21 128.58      89.755769       NaN            0
   구미 36.13 128.34      89.992145       NaN            0
   예천 36.65 128.46      90.311043       NaN            0
   영동 37.18 128.46      90.547691       NaN            0

분류: daylight 0 / dusk 9 / 일몰 후 2


→ **모델 함의**: 같은 timestamp에 사이트별 day/dusk/dark 갈림 확인. 모델이 site_emb × zenith 결합으로 자연스럽게 학습 가능 (FiLM 도움 없이도 zenith feature 자체가 spatial 정보 인코딩).

## 7. 학습 mask 정책 — 단순화 확정

In [8]:
# train_mask: dsr_mean이 살아있으면 학습, 없으면 제외
df['train_mask'] = df.dsr_mean.notna().astype(int)

print('--- train_mask 분포 ---')
print(df.train_mask.value_counts())
print()
n_total = len(df)
n_train = (df.train_mask == 1).sum()
print(f'학습 행: {n_train:,} ({n_train/n_total*100:.1f}%)')
print(f'제외 행: {n_total - n_train:,} ({(n_total-n_train)/n_total*100:.1f}%)')
print()
print('--- mask=0 행의 zenith 분포 ---')
print(df[df.train_mask == 0]['zenith_center'].describe().round(2))
print('(거의 모두 zenith>=90°이면 야간 trivial — 정상)')

--- train_mask 분포 ---
train_mask
0    206836
1    178659
Name: count, dtype: int64

학습 행: 178,659 (46.3%)
제외 행: 206,836 (53.7%)

--- mask=0 행의 zenith 분포 ---
count    206836.00
mean        120.11
std          21.05
min          81.75
25%         102.92
50%         119.03
75%         135.41
max         168.87
Name: zenith_center, dtype: float64
(거의 모두 zenith>=90°이면 야간 trivial — 정상)


→ **모델 함의**: mask=0 = 야간/sat_outage. PV target=0이고 input(dsr)도 NaN이라 학습 시 backprop 깨짐 방지용. magic number weight(0.5 등) 폐기, binary mask로 충분. site×time 변동성은 모델(FiLM-GRU)이 zenith·site_emb feature로 자동 학습.

## 8. 의사결정 요약

**확정**:
1. **데이터 단위**: 사이트 × 시간 (`datetime_kst` hour-ending 라벨) — KOEN PV target과 자연 정합
2. **유효 학습 데이터**: 11 사이트 × 35,045 시간 ≈ 385k 행 중 `dsr_mean.notna()`인 행만 (~42%)
3. **mask 정책**: `train_mask = dsr_mean.notna()` (binary)
   - Track A (XGBoost): mask=0 행 drop
   - Track B (FiLM-GRU): mask=0 행 유지 + loss 곱셈자
4. **status 분류 폐기**: zenith를 직접 모델 feature로 사용 (continuous). 4-tier categorical 분류는 EDA 진단 용도로만 사용, 학습 입력엔 안 씀.
5. **품질 weight 폐기**: 우리 데이터에서 dsr_dqf binary, sw_dqf=zenith 함수, ASR/RSR 미사용 → 품질 차원 가중치 근거 없음.

**모델 입력 후보 컬럼**:
- 학습 input: `dsr_mean`, `zenith_center`, `lat`, `lon`, `site_id`, `hour`, `month`, `dsr_n_valid` (옵션)
- 학습 target: KOEN solar_hourly의 시간 PV (별도 join)
- 학습 mask: `train_mask`

**다음 단계**:
- `01_gk2a_v2_diagnostics.ipynb`: 사이트별·계절별 DSR 패턴, zenith-DSR clearsky 정합 진단
- `02_pv_target_join.ipynb`: GK-2A v2 + KOEN solar_hourly 시간 정렬 + 호기→사이트 매핑
- `07_om_forecast_inspect.ipynb` (`plan/pv/plan_v1.md` Gate 1): Open-Meteo `historical-forecast-api` dawn/dusk GHI 커버리지 검증